# Lab 1 — Gemini API basics

Notes to self while exploring the `google-genai` SDK: interactions, streaming, multi-turn context, and structured (JSON) output with Pydantic.

Needs a `.env` with `GEMINI_API_KEY=...`.

---
## 1. Load the API key

Keep the key in `.env` (git-ignored) and load it into env vars rather than hard-coding it.

In [1]:
# Step 1: Load the API keys into environment variables from the .env file
from dotenv import load_dotenv

print("Loading API keys from .env file...")

# Set override=True to ensure that the .env file values take precedence over existing environment variables
load_dotenv(override=True) 

Loading API keys from .env file...


True

In [2]:
# Step 2: Access the Gemini API key from the environment variables
import os

gemini_api_key = os.getenv("GEMINI_API_KEY")

if gemini_api_key:
    print(f"Gemini API key loaded successfully and begins {gemini_api_key[:4]}")
else:
    print("Gemini API key not found. Please check your .env file.")

Gemini API key loaded successfully and begins AQ.A


In [3]:
# Step 3: Import libraries
from google import genai
from data_science.gemini.types import GeminiModelType

print("Libraries imported successfully.")
print("Starting the Gemini API interaction...")
print("="*60)

Libraries imported successfully.
Starting the Gemini API interaction...


---
## 2. Initialize the client

`genai.Client()` reads `GEMINI_API_KEY` from the env automatically.


In [4]:
# Initialize the client
client = genai.Client()

---
## 3. First interaction

One request/response = an *interaction*. 
The API returns the full Interaction resource containing metadata, usage statistics, and the step-by-step history of the turn.
While the SDKs expose the full response, they also provide convenience properties like interaction.output_text and interaction.output_image to access final outputs directly.

https://ai.google.dev/gemini-api/docs/interactions-overview

In [5]:
# Create an interaction with the Gemini API
interaction = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="Explain how AI works in few sentences"
)
print(interaction)




status='completed' model='gemini-3.5-flash-lite' agent=None id='v1_Chdpc0IxYXJ5T0hxXzcxZThQb1BtdndRcxIXaXNCMWFyeU9IcV83MWU4UG9QbXZ3UXM' created='2026-08-07T11:24:58Z' updated='2026-08-07T11:24:58Z' system_instruction=None tools=None usage=Usage(cached_tokens_by_modality=None, grounding_tool_count=None, input_tokens_by_modality=[ModalityTokens(modality='text', tokens=8)], output_tokens_by_modality=None, tool_use_tokens_by_modality=None, total_cached_tokens=0, total_input_tokens=8, total_output_tokens=69, total_thought_tokens=0, total_tokens=77, total_tool_use_tokens=0, raw_prompt_token=39) response_modalities=None response_mime_type=None previous_interaction_id=None environment_id=None service_tier='standard' webhook_config=None steps=[ThoughtStep(signature='EjQKMgERTTIPZv0s5d841CzfvGSguxHm2tmDgMrkoA7a6lPK0kj4vDnxvySM1cvN3jikJdTy', summary=None, type='thought'), ModelOutputStep(content=[TextContent(text='Artificial Intelligence (AI) works by feeding massive amounts of data into computer

In [6]:
interaction.output_text

'Artificial Intelligence (AI) works by feeding massive amounts of data into computer algorithms that look for patterns and relationships. Using this data, the system builds a mathematical model to make predictions, solve problems, or generate new content. Instead of following rigid, pre-written code, the AI continuously learns and improves its accuracy by analyzing new information and feedback.'

---
## 4. Streaming

`stream=True`

In [7]:
# Stream the response
stream = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="Explain how AI works in few sentences",
    stream=True
)

for event in stream:
    print(event)

interaction=InteractionSseEventInteraction(id='v1_ChdqY0IxYXQtSEtiVFExZThQaXNmd2lBMBIXamNCMWF0LUhLYlRRMWU4UGlzZndpQTA', status='in_progress', object='interaction', model='gemini-3.5-flash-lite', agent=None, created=None, updated=None, service_tier=None, usage=None, steps=None) event_type='interaction.created' event_id=None metadata=None
interaction_id='v1_ChdqY0IxYXQtSEtiVFExZThQaXNmd2lBMBIXamNCMWF0LUhLYlRRMWU4UGlzZndpQTA' status='in_progress' event_id=None event_type='interaction.status_update' metadata=None
index=0 step=ThoughtStep(signature=None, summary=None, type='thought') event_id=None event_type='step.start' metadata=None
delta=ThoughtSignatureDelta(signature='EjQKMgERTTIPsbonPSTq15f4giOvCQi0tBLc6tKo7R9kW06uGeu+7gu6741SskJ29Q1A+5Lf', type='thought_signature') index=0 event_id=None event_type='step.delta' metadata=None
index=0 event_id=None event_type='step.stop' metadata=None step_usage=None usage=None
index=1 step=ModelOutputStep(content=None, error=None, type='model_output') 

---
## 5. Multi-turn
Continue a conversation on the server using `previous_interaction_id`

In [8]:
# Multi-turn conversation
# See more in https://ai.google.dev/gemini-api/docs/text-generation#multi-turn-conversations
interaction1 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="I have two dogs in my house."
)
print(f"Response 1: {interaction1.output_text}")

interaction2 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="How many paws are in my house?",
    previous_interaction_id=interaction1.id
)
print(f"Response 2: {interaction2.output_text}")

Response 1: Aw, that's wonderful! What kind of dogs are they? Are they high-energy puppies or more laid-back seniors? 

Tell me their names if you'd like!
Response 2: Assuming they are both typical dogs with four paws each, and assuming no other pets or humans with extra limbs...

There are **8** paws in your house! 

*(Plus, of course, your feet, unless you count those differently!)*


In [9]:
# Multi-turn conversation with a streamed follow-up
interaction1 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="I have 2 dogs in my house.",
)
print(interaction1.output_text)

# Link this turn to the previous one so the model keeps the conversation context
stream = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="How many paws are in my house?",
    previous_interaction_id=interaction1.id,
    stream=True
)
# Keep only the text deltas and print them incrementally (skip lifecycle/thought events)
for event in stream:
    if event.event_type == "step.delta":
        if event.delta.type == "text":
            print(event.delta.text, end="")

That's awesome! What kind of dogs are they, and what are their names?
Assuming all your dogs have the usual number of paws (four each), there are **8 dog paws** in your house. 

Depending on how many humans live there too, you can add another 2 feet (or hands/feet combined!) per person to the total!

<h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try this commercial application:<br/>
            First ask the LLM to pick a business area that might be worth exploring for an Agentic AI opportunity.<br/>
            Then ask the LLM to present a pain-point in that industry - something challenging that might be ripe for an Agentic solution.<br/>
            Finally have 3 third LLM call propose the Agentic AI solution. <br/>
            We will cover this at up-coming labs, so don't worry if you're unsure.. just give it a try!
            </span>

---
## 6. Structured output (Pydantic)

In [10]:
from pydantic import BaseModel, Field
class BusinessArea(BaseModel):
    business_area: str = Field(description="A business area that might be worth exploring for an Agentic AI opportunity.")
    description: str = Field(description="A brief description of this business area.")
    reasoning: str = Field(description="The reasoning behind why this business area is worth exploring for an Agentic AI opportunity.")

interaction1 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input=(
        "Pick a single, high-value business area or industry that is "
        "worth exploring for an Agentic AI opportunity. Name the industry "
        "and explain in 2-3 sentences why it is a strong candidate."
    ),
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": BusinessArea.model_json_schema()
    },
)

interaction1.output_text

'{\n  "business_area": "Clinical Trial Operations in Life Sciences",\n  "description": "The end-to-end management of pharmaceutical clinical trials, encompassing patient recruitment, site monitoring, regulatory compliance, and protocol deviation management.",\n  "reasoning": "This area is a prime candidate for Agentic AI because trial execution involves complex, multi-step workflows across disparate stakeholders that currently suffer from severe bottlenecks and costly delays. Autonomous agents can continuously monitor data streams, proactively resolve site issues, and dynamically optimize patient matching, drastically reducing time-to-market for life-saving therapeutics."\n}'

In [11]:
business_area = BusinessArea.model_validate_json(interaction1.output_text)
business_area


BusinessArea(business_area='Clinical Trial Operations in Life Sciences', description='The end-to-end management of pharmaceutical clinical trials, encompassing patient recruitment, site monitoring, regulatory compliance, and protocol deviation management.', reasoning='This area is a prime candidate for Agentic AI because trial execution involves complex, multi-step workflows across disparate stakeholders that currently suffer from severe bottlenecks and costly delays. Autonomous agents can continuously monitor data streams, proactively resolve site issues, and dynamically optimize patient matching, drastically reducing time-to-market for life-saving therapeutics.')

In [12]:
class PainPoint(BaseModel):
    pain_point: str = Field(description="A pain point for the business area.")
    description: str = Field(description="A brief description of why this pain point.")

interaction2 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="In the business area selected, present a single, high-value pain point that an Agentic AI could address. Explain in 2-3 sentences why this pain point is significant.",
    previous_interaction_id=interaction1.id,
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": PainPoint.model_json_schema()
    },
)
interaction2.output_text

'{\n  "pain_point": "Patient Recruitment and Retention Bottlenecks",\n  "description": "Clinical trials frequently fail or face massive delays because sites struggle to identify, screen, and retain eligible patient cohorts against strict inclusion criteria. This pain point is significant because recruitment delays cost pharmaceutical companies up to $8 million per day in lost revenue, while high dropout rates compromise the statistical validity of the entire trial."\n}'

In [13]:
pain_point = PainPoint.model_validate_json(interaction2.output_text)
pain_point

PainPoint(pain_point='Patient Recruitment and Retention Bottlenecks', description='Clinical trials frequently fail or face massive delays because sites struggle to identify, screen, and retain eligible patient cohorts against strict inclusion criteria. This pain point is significant because recruitment delays cost pharmaceutical companies up to $8 million per day in lost revenue, while high dropout rates compromise the statistical validity of the entire trial.')

In [14]:
class Solution(BaseModel):
    solution: str = Field(description="A solution for the pain point.")
    description: str = Field(description="A brief description of this solution")
    reasoning: str = Field(description="A brief description of the reasoning behind this solution.")
    
interaction3 = client.interactions.create(
    model=GeminiModelType.GEMINI_35_FLASH_LITE,
    input="Propose an Agentic AI solution for this pain point for the selected business area. Explain in 2-3 sentences why this solution is effective.",
    previous_interaction_id=interaction2.id,
    response_format= {
        "type": "text",
        "mime_type": "application/json",
        "schema": Solution.model_json_schema()
    },
)
interaction3.output_text

'{\n  "solution": "Autonomous Patient Matching and Engagement Swarm",\n  "description": "An orchestrated multi-agent system that continuously ingests real-world electronic health record (EHR) data to identify eligible trial candidates, autonomously drafts compliant outreach messaging for physicians, and deploys empathetic conversational agents to guide patients through pre-screening and schedule follow-ups.",\n  "reasoning": "This solution is effective because it shifts patient recruitment from a reactive, manual outreach model to a proactive, 24/7 automated engine capable of instantly matching complex genomic and clinical criteria. Furthermore, the continuous engagement agents build trust and address patient friction points in real-time, dramatically cutting dropout rates and accelerating trial timelines."\n}'

In [15]:
solution = Solution.model_validate_json(interaction3.output_text)
print(solution.solution)
print(solution.description)

Autonomous Patient Matching and Engagement Swarm
An orchestrated multi-agent system that continuously ingests real-world electronic health record (EHR) data to identify eligible trial candidates, autonomously drafts compliant outreach messaging for physicians, and deploys empathetic conversational agents to guide patients through pre-screening and schedule follow-ups.


---
## 7. Final report

Print the three validated objects together.

In [16]:
# Print the three chained results as a single, readable report
print(f"Business Area: {business_area.business_area}")
print(f"\t - {business_area.description}")
print(f"\t - {business_area.reasoning}")
print(f"\nPain Point: {pain_point.pain_point}")
print(f"\t - {pain_point.description}")
print(f"\nSolution: {solution.solution}")
print(f"\t - {solution.description}")

Business Area: Clinical Trial Operations in Life Sciences
	 - The end-to-end management of pharmaceutical clinical trials, encompassing patient recruitment, site monitoring, regulatory compliance, and protocol deviation management.
	 - This area is a prime candidate for Agentic AI because trial execution involves complex, multi-step workflows across disparate stakeholders that currently suffer from severe bottlenecks and costly delays. Autonomous agents can continuously monitor data streams, proactively resolve site issues, and dynamically optimize patient matching, drastically reducing time-to-market for life-saving therapeutics.

Pain Point: Patient Recruitment and Retention Bottlenecks
	 - Clinical trials frequently fail or face massive delays because sites struggle to identify, screen, and retain eligible patient cohorts against strict inclusion criteria. This pain point is significant because recruitment delays cost pharmaceutical companies up to $8 million per day in lost revenue